# 🟢 2.5 Time Travel & State Recovery: Checkpoints, `/rewind`, and `/branch` Exploration
**Course:** Agentic CLI Engineering - Complete Claude Code Masterclass  
**Module:** Section 2 — Interactive Session Mastery & Context Hygiene  
**Shared Section 2 Project:** `./order_service` (`v2.4-gearbox-and-verified` $\rightarrow$ `v2.5-section2-complete`)  
**Configured OpenRouter Model:** `deepseek/deepseek-v4.1-flash-1017`  

---

## 🏛️ Architectural Objective
When a bold architectural refactor goes sideways, using traditional `git reset` in your terminal leaves a **Split-Brain AI**: your files roll back, but Claude Code's internal conversation memory still remembers the failed code, causing hallucinations on subsequent turns.

Claude Code solves this via **Atomic State Rewinding (`/rewind` / double-`Esc`)**, which rolls back both mutated disk files *and* LLM conversation history simultaneously.

In this hands-on lab, you will:
1. **Stage a Clean REST Endpoint Baseline (`rest_service.py`)** inside our shared `./order_service` repository and lock a Git checkpoint (`v2.5-pre-refactor-checkpoint`).
2. **Simulate a Destructive GraphQL Refactor** that breaks core invariants and causes test suite failures.
3. **Execute Atomic `/rewind` State Recovery** (in your **Claude CLI Terminal** and via our deterministic rollback harness) to purge both corrupted disk files and dead-end agent memory in one operation.
4. **Execute a Clean REST Optimization Branch** and verify convergence across the entire Section 2 `./order_service` test suite via `pytest` (`Exit Code 0`).

---
## Phase 1: Staging the Clean REST Endpoint Checkpoint in `./order_service`
Let's prepare our shared `./order_service` workspace, ensure `.claude/settings.json` is bound to `deepseek/deepseek-v4.1-flash-1017`, create our clean REST order payload endpoint (`rest_service.py`) and its baseline test suite (`test_rest_service.py`), and commit it to Git as our immutable checkpoint baseline (`v2.5-pre-refactor-checkpoint`).

In [ ]:
%%bash
# 0. Auto-sync required packages from root requirements.txt
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
REQ_FILE=$( [ -f "requirements.txt" ] && echo "requirements.txt" || echo "../requirements.txt" )
if [ -f "$REQ_FILE" ]; then
    echo -e "\033[0;36m[SETUP] Syncing Python dependencies from $REQ_FILE...\033[0m"
    $PY_BIN -m pip install -q -r "$REQ_FILE"
fi

# Self-healing guard if ./order_service was not initialized yet
if [ ! -d "order_service/.git" ]; then
    mkdir -p order_service/.claude
    cd order_service
    printf "__pycache__/\n*.pyc\n.pytest_cache/\n.env*\n.claude/\n" > .gitignore
    git init -q
    git config user.email "architect@fnosystematics.co.in"
    git config user.name "Agentic Architect"
    git add . && git commit -q -m "chore(init): initialize shared order_service repository"
    cd ..
fi

mkdir -p order_service/.claude
cd order_service

SAMPLE_SETTINGS="../../Foundation & Memory Architecture/lab_1_2_workspace/.claude_sample/settings.json"
if [ -f "$SAMPLE_SETTINGS" ]; then
    cp "$SAMPLE_SETTINGS" .claude/settings.json
fi

# Clean any leftover untracked experimental files from prior runs
rm -f graphql_adapter.py

# 1. Write Clean REST Service
cat << 'EOF' > rest_service.py
def get_order_payload(order_id: str) -> dict:
    return {
        "order_id": order_id,
        "status": "CONFIRMED",
        "items": [{"sku": "SKU-99", "qty": 2, "price_cents": 1500}]
    }
EOF

# 2. Write Baseline Test Suite
cat << 'EOF' > test_rest_service.py
from rest_service import get_order_payload

def test_clean_rest_payload():
    res = get_order_payload("ORD-500")
    assert res["order_id"] == "ORD-500"
    assert res["status"] == "CONFIRMED"
    assert len(res["items"]) == 1
EOF

git add rest_service.py test_rest_service.py
git commit -q -m "chore(rest): clean baseline checkpoint" || true
git tag -f v2.5-pre-refactor-checkpoint

echo -e "\033[1;32m[✔] Baseline REST Service Staged & Committed in ./order_service (HEAD: $(git rev-parse --short HEAD))\033[0m"
echo -e "\033[1;32m[✔] Active OpenRouter Model: deepseek/deepseek-v4.1-flash-1017\033[0m"

---
## Phase 2: Simulating the Destructive GraphQL Refactor
Suppose an agent attempts a radical refactor inside `./order_service`, introducing an over-engineered, unverified GraphQL resolver (`graphql_adapter.py`) that breaks our test contract (`test_rest_service.py`) and throws runtime schema validation errors.

Let's apply this broken refactor and run `pytest` to see the failure.

In [ ]:
%%writefile order_service/graphql_adapter.py
# ❌ FAILED REFACTOR: Over-engineered GraphQL adapter breaking existing REST contract
class BrokenGraphQLAdapter:
    def __init__(self, schema_registry: dict):
        self.registry = schema_registry

    def resolve(self, query: str) -> None:
        # Throws KeyError because expected GraphQL schema types are missing
        raise KeyError("GraphQL schema violation: type 'Query' not found in registry")


In [ ]:
%%writefile order_service/test_rest_service.py
# Test suite impacted by broken GraphQL refactor attempt
from graphql_adapter import BrokenGraphQLAdapter

def test_broken_graphql_refactor():
    adapter = BrokenGraphQLAdapter({})
    adapter.resolve("{ order(id: 'ORD-500') { status } }")


In [ ]:
%%bash
cd order_service
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

echo -e "\033[1;31m[TEST RUN] Verifying failed GraphQL refactor state...\033[0m"
$PY_BIN -m pytest test_rest_service.py -q --tb=short || true

---
## Phase 3: 🖥️ Execute Atomic `/rewind` State Recovery in Your Claude CLI Terminal
We have hit a dead end. If we only run `git checkout .` in a bash shell, our files revert, but Claude Code's session memory would still retain `graphql_adapter.py` context.

### Interactive `/rewind` & `/branch` in Your Claude CLI Terminal
Open your **Claude CLI Terminal** inside `./order_service` to trigger interactive time travel:
```bash
# 1. Enter the shared Section 2 project directory
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"

# 2. Launch Claude Code with deepseek/deepseek-v4.1-flash-1017
claude --model "deepseek/deepseek-v4.1-flash-1017"

# 3. Inside Claude Code, press Esc twice (or type /rewind) to open the Checkpoint Time-Travel Menu:
> /rewind
#    Select: "Restore code and conversation" at the pre-GraphQL checkpoint!
```

Below, we also run our deterministic `state_rewind_engine.py` recovery harness inside `./order_service`, which atomically purges the failed `graphql_adapter.py` refactor from disk *and* restores the working tree back to the `v2.5-pre-refactor-checkpoint` baseline.

In [ ]:
%%writefile order_service/state_rewind_engine.py
import subprocess
from pathlib import Path

def execute_atomic_rewind() -> None:
    """
    Simulates Claude Code's atomic /rewind (double-Esc):
    1. Rolls back tracked disk files via git checkout to the last checkpoint.
    2. Purges dead-end experimental files created during the failed refactor.
    """
    # 1. Atomic disk rollback of tracked files
    subprocess.run(["git", "checkout", "-q", "HEAD", "--"], check=True)

    # 2. Purge dead-end experimental files created during refactor
    graphql_file = Path("graphql_adapter.py")
    if graphql_file.exists():
        graphql_file.unlink()

if __name__ == "__main__":
    execute_atomic_rewind()
    print("\033[1;32m[✔] ATOMIC '/rewind' COMPLETE: Disk files & memory state restored to baseline checkpoint.\033[0m")


In [ ]:
%%bash
cd order_service
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

echo -e "\033[1;32m[TIME TRAVEL] Executing atomic /rewind state recovery...\033[0m"
$PY_BIN state_rewind_engine.py

echo ""
echo -e "\033[1;32m[CHECKPOINT AUDIT] Verifying restored baseline test suite:\033[0m"
$PY_BIN -m pytest test_rest_service.py -q
git log -1 --oneline

---
## Phase 4: Executing a Clean REST Optimization Branch & Verifying All Section 2 Tests via `pytest`
Now that we have successfully rewound to our pristine baseline without any LLM memory pollution, we apply a clean, non-destructive optimization to `rest_service.py`: adding a cryptographic SHA-256 response checksum to our REST payload, and running the **entire Section 2 `pytest` suite** across `./order_service` (`2.2` + `2.3` + `2.4` + `2.5`)!

In [ ]:
%%writefile order_service/rest_service.py
# ✔ CLEAN RESTART: Non-destructive REST optimization (Response Checksum)
import hashlib
import json

def get_order_payload(order_id: str) -> dict:
    payload = {
        "order_id": order_id,
        "status": "CONFIRMED",
        "items": [{"sku": "SKU-99", "qty": 2, "price_cents": 1500}]
    }
    canonical_json = json.dumps(payload, sort_keys=True).encode("utf-8")
    payload["checksum"] = hashlib.sha256(canonical_json).hexdigest()[:12]
    return payload


In [ ]:
%%writefile order_service/test_rest_service.py
# ✔ Clean test suite verifying optimized REST service
from rest_service import get_order_payload

def test_optimized_rest_payload_with_checksum():
    res = get_order_payload("ORD-500")
    assert res["order_id"] == "ORD-500"
    assert res["status"] == "CONFIRMED"
    assert "checksum" in res
    assert len(res["checksum"]) == 12


In [ ]:
%%bash
cd order_service
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

echo -e "\033[1;32m=================================================================\033[0m"
echo -e "\033[1;32m [PHASE 4] PYTEST VERIFICATION OF OPTIMIZED REST & SECTION 2     \033[0m"
echo -e "\033[1;32m=================================================================\033[0m"
$PY_BIN -m pytest -v

# Commit and tag the final Section 2 milestone in ./order_service
git add .
git commit -q -m "feat(rest): add SHA-256 payload checksum after atomic /rewind recovery" || true
git tag -f v2.5-section2-complete
echo -e "\n\033[1;32m[SECTION 2 GIT MILESTONES LOCKED IN ./order_service]:\033[0m"
git tag -l "v2.*"